# Task 4

1. Implement RNN model

    - Choose one of the tasks, either tagging or text classification or __generating__ next token.

    - Build and train RNN model that learns the chosen task.

    - Make sure to have a layer of embedding, at least one RNN layer and linear transformation before output.

    - Demonstrate that model works (on any examples).

- __(Optional)__ Implement separately embedding training module and use pre-trained embeddings instead of embedding layer before RNN.

In [2]:
import os
import torch
import spacy
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

In [3]:
base_path = os.getcwd()
file_path = os.path.abspath(os.path.join(base_path, "..", "data", "without_chap_and_title", "eng_Anne_full_abbr.txt"))
text = open(file_path, encoding="utf-8").read()

In [4]:
# Токенизация текста с использованием spaCy
nlp = spacy.load("en_core_web_sm")

In [5]:
def tokenize_text(text):
    doc = nlp(text)
    return [token.text.lower() for token in doc if not token.is_punct and not token.is_space]

tokens = tokenize_text(text)

In [6]:
# сreating a vocabulary
vocab = set(tokens)
word2idx = {word: idx for idx, word in enumerate(vocab)}
idx2word = {idx: word for word, idx in word2idx.items()}
vocab_size = len(word2idx)

In [7]:
# training CBOW for pre-trained embeddings 
class CBOW(nn.Module):
    def __init__(self, vocab_size, embedding_dim):
        super(CBOW, self).__init__()
        self.embeddings = nn.Embedding(vocab_size, embedding_dim)
        self.linear = nn.Linear(embedding_dim, vocab_size)
    
    def forward(self, context_idxs):
        embeds = self.embeddings(context_idxs).mean(dim=1)
        out = self.linear(embeds)
        return out

In [8]:
# dataset to CBOW
class CBOWDataset(Dataset):
    def __init__(self, tokens, context_size):
        self.data = []
        for i in range(context_size, len(tokens) - context_size):
            context = tokens[i - context_size:i] + tokens[i + 1:i + context_size + 1]
            target = tokens[i]
            self.data.append((context, target))
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        context, target = self.data[idx]
        context_idxs = torch.tensor([word2idx[w] for w in context])
        target_idx = torch.tensor(word2idx[target])
        return context_idxs, target_idx

In [9]:
embedding_dim = 128
context_size = 2
cbow_dataset = CBOWDataset(tokens, context_size)
cbow_loader = DataLoader(cbow_dataset, batch_size=64, shuffle=True)

cbow_model = CBOW(vocab_size, embedding_dim)
cbow_criterion = nn.CrossEntropyLoss()
cbow_optimizer = optim.Adam(cbow_model.parameters(), lr=0.001)

In [10]:
# training CBOW
for epoch in range(200):
    total_loss = 0
    for context_idxs, target_idx in cbow_loader:
        cbow_optimizer.zero_grad()
        output = cbow_model(context_idxs)
        loss = cbow_criterion(output, target_idx)
        loss.backward()
        cbow_optimizer.step()
        total_loss += loss.item()
    print(f"[CBOW] Epoch {epoch+1}, Loss: {total_loss / len(cbow_loader):.4f}")

[CBOW] Epoch 1, Loss: 6.6298
[CBOW] Epoch 2, Loss: 5.5072
[CBOW] Epoch 3, Loss: 5.0336
[CBOW] Epoch 4, Loss: 4.6841
[CBOW] Epoch 5, Loss: 4.3984
[CBOW] Epoch 6, Loss: 4.1541
[CBOW] Epoch 7, Loss: 3.9397
[CBOW] Epoch 8, Loss: 3.7501


KeyboardInterrupt: 

In [ ]:
#  use of pre-trained embeddings in RNN
pretrained_embeddings = cbow_model.embeddings.weight.data.clone()

class TokenGeneratorRNN(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, pretrained_embeddings):
        super(TokenGeneratorRNN, self).__init__()
        self.embeddings = nn.Embedding.from_pretrained(pretrained_embeddings, freeze=False)
        self.rnn = nn.RNN(embedding_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, vocab_size)
    
    def forward(self, x):
        embeds = self.embeddings(x)
        rnn_out, _ = self.rnn(embeds)
        output = self.fc(rnn_out[:, -1, :])  # последний токен последовательности
        return output

In [ ]:
# use of pre-trained embeddings in RNN
class TokenDataset(Dataset):
    def __init__(self, tokens, seq_length):
        self.tokens = tokens
        self.seq_length = seq_length
        self.data = self.create_sequences()
    
    def create_sequences(self):
        sequences = []
        for i in range(len(self.tokens) - self.seq_length):
            input_seq = self.tokens[i:i + self.seq_length]
            target = self.tokens[i + self.seq_length]
            sequences.append((input_seq, target))
        return sequences
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        input_seq, target = self.data[idx]
        input_seq_idx = torch.tensor([word2idx[word] for word in input_seq])
        target_idx = torch.tensor(word2idx[target])
        return input_seq_idx, target_idx

In [ ]:
seq_length = 5
dataset = TokenDataset(tokens, seq_length)
dataloader = DataLoader(dataset, batch_size=64, shuffle=True)

hidden_dim = 256
model = TokenGeneratorRNN(vocab_size, embedding_dim, hidden_dim, pretrained_embeddings)

loss_fn = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [ ]:
# RNN training
for epoch in range(10):
    total_loss = 0
    for input_seq, target in dataloader:
        optimizer.zero_grad()
        output = model(input_seq)
        loss = loss_fn(output, target)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"[RNN] Epoch {epoch+1}, Loss: {total_loss / len(dataloader):.4f}")

In [ ]:
# next token generation
def generate_next_token(model, input_text, word2idx, idx2word, seq_length):
    model.eval()
    input_tokens = tokenize_text(input_text)
    input_idx = [word2idx[word] for word in input_tokens[-seq_length:]]
    input_tensor = torch.tensor(input_idx).unsqueeze(0)
    
    with torch.no_grad():
        output = model(input_tensor)
    predicted_idx = torch.argmax(output, dim=1).item()
    return idx2word[predicted_idx]

In [ ]:
input_text = "she was feeling"
predicted_token = generate_next_token(model, input_text, word2idx, idx2word, seq_length)
print(f"Next token after '{input_text}': {predicted_token}")

[CBOW] Epoch 1, Loss: 6.6393
[CBOW] Epoch 2, Loss: 5.5066
[CBOW] Epoch 3, Loss: 5.0358
[CBOW] Epoch 4, Loss: 4.6895
[CBOW] Epoch 5, Loss: 4.4060
[RNN] Epoch 1, Loss: 5.7544
[RNN] Epoch 2, Loss: 4.9205
[RNN] Epoch 3, Loss: 4.3892
[RNN] Epoch 4, Loss: 3.9378
[RNN] Epoch 5, Loss: 3.5462
[RNN] Epoch 6, Loss: 3.2115
[RNN] Epoch 7, Loss: 2.9221
[RNN] Epoch 8, Loss: 2.6697
[RNN] Epoch 9, Loss: 2.4471
[RNN] Epoch 10, Loss: 2.2497
Next token after 'she was feeling': that
